In [1]:
from pathlib import Path
import dataclasses
import pickle
import numpy as np

from flowing import flowtime_to_radius
from measurements import *


In [2]:
ns = 64
nt = 16

flowtimes = [ 0.02572923590301565, 0.05303278154520656, 0.08103495224784385, 0.110208782326772, 0.141058391298962, 0.1736893835495303,
              0.2086835114617738, 0.2454549361253429, 0.2838117228448263, 0.3249152519239425, 0.3705101434001126, 0.4179449298122797,
              0.4696233239687699, 0.5279559816139976, 0.5890993819917063, 0.6570215111572449, 0.734680305158274, 0.8257738792221512,
              0.9353328035912798, 1.06959076807306, 1.233820281085228, 1.435936459574629 ]#, 1.68034447439404, 1.972161672512587 ]
print(', '.join(f'{flowtime_to_radius(ft, nt):.3f}' for ft in flowtimes))

local_tol = 3.00e-04    # flowtimes based on this (not used here)
flow_type = 'zeuthen'   # just for information, prabably not needed

0.028, 0.041, 0.050, 0.059, 0.066, 0.074, 0.081, 0.088, 0.094, 0.101, 0.108, 0.114, 0.121, 0.128, 0.136, 0.143, 0.152, 0.161, 0.171, 0.183, 0.196, 0.212


In [3]:
scratch_path = Path('/work/scratch/ln29bamu')
home_path = Path('/home/ln29bamu')
code_path = home_path / 'code' / 'milc_luis'

flowfiles_folder = scratch_path / 'flow_files' / 'combined'

flowmeas_config = [ parse_flow_output(fn.read_text(), flowtimes)
                    for fn in flowfiles_folder.iterdir() ]

print(len(flowmeas_config))

3055


In [4]:
# save all Measurement objects (with q-corrs removed)

flowmeas_nocorrs_config = [ [ dataclasses.replace(meas, q_corrs=None)
                              for meas in flowmeas ]
                            for flowmeas in flowmeas_config ]

pickle_file = code_path / 'zeugs' / 'outputs' / '16x64flows' / 'data0a.pkl'

with pickle_file.open('wb') as f:
    pickle.dump(flowmeas_nocorrs_config, f)

In [5]:
# include all configs, that is also potentially unthermalized
skip_configs = 0

n_cofigs    = len(flowmeas_config[skip_configs:])
n_flowtimes = len(flowmeas_config[0])
n_tau       = len(flowmeas_config[0][0].q_corrs)
n_r2        = len(flowmeas_config[0][0].q_corrs[0])

all_corrs_r2 = np.empty(shape=(n_cofigs, n_flowtimes, n_tau, n_r2), dtype=float)

for i_cfg, cfg in enumerate(flowmeas_config[skip_configs:]):
    for i_flow, flowmeas in enumerate(cfg):
        all_corrs_r2[i_cfg, i_flow, :, :] = np.array(flowmeas.q_corrs)

print(all_corrs_r2.shape, all_corrs_r2.dtype)

(3055, 22, 9, 3073) float64


In [6]:
dr2_list = radial_multiplicities(ns)
r_list   = radial_separations(ns)
n_r      = len(r_list)

all_corrs = np.empty(shape=(n_cofigs, n_flowtimes, n_tau, n_r))

for i_cfg in range(n_cofigs):
    for i_flow in range(n_flowtimes):
        for i_tau in range(n_tau):
            all_corrs[i_cfg, i_flow, i_tau, :] = np.array(make_distance_corr_arrays( all_corrs_r2[i_cfg, i_flow, i_tau, :],
                                                                                     True,
                                                                                     dr2_list ))
            
print(all_corrs.shape, all_corrs.dtype)
print('total numbers =', np.prod(all_corrs.shape))

(3055, 22, 9, 1914) float64
total numbers = 1157759460


In [7]:
# save array to file
np.save(
    code_path / 'zeugs' / 'outputs' / '16x64flows' / 'data0.npy',
    all_corrs
)